In [1]:
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

# 加載鳶尾花資料集
iris = datasets.load_iris()
X = iris.data
y = iris.target

# 為了簡化問題，我們只使用前兩個特徵，並且只考慮兩個類別（二元分類）
X = X[y != 2, :2]
y = y[y != 2]

# 分割訓練集和測試集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 特徵標準化
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train.shape, X_test.shape, y_train.shape, y_test.shape



((70, 2), (30, 2), (70,), (30,))

In [7]:
# 確認 polynomial_kernel 函數 K(x,y) = (x·y + c)^d
def polynomial_kernel(x, y, degree=3, coef=1):
    dot = np.dot(x, y)
    return (dot + coef) ** degree

# 測試多項式核函數
test_x = np.array([1, 2])
test_y = np.array([3, 4])
polynomial_kernel(test_x, test_y, degree=3, coef=1)

1728

In [4]:
class SimplifiedSVM:
    def __init__(self, C=1.0, max_iter=100, kernel=polynomial_kernel, degree=3):
        self.C = C
        self.max_iter = max_iter
        self.kernel = kernel
        self.degree = degree
        self.alpha = None
        self.b = 0

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.alpha = np.zeros(n_samples)
        
        # Kernel matrix
        K = np.zeros((n_samples, n_samples))
        for i in range(n_samples):
            for j in range(n_samples):
                K[i, j] = self.kernel(X[i], X[j], degree=self.degree)

        for _ in range(self.max_iter):
            for i in range(n_samples):
                error_i = self.decision_function(X[i]) - y[i]
                if (y[i] * error_i < -1e-3 and self.alpha[i] < self.C) or (y[i] * error_i > 1e-3 and self.alpha[i] > 0):
                    j = np.random.randint(0, n_samples)
                    while j == i:
                        j = np.random.randint(0, n_samples)
                    error_j = self.decision_function(X[j]) - y[j]

                    # Save old alphas
                    alpha_i_old, alpha_j_old = self.alpha[i], self.alpha[j]

                    # Compute new alphas
                    if y[i] != y[j]:
                        L = max(0, self.alpha[j] - self.alpha[i])
                        H = min(self.C, self.C + self.alpha[j] - self.alpha[i])
                    else:
                        L = max(0, self.alpha[i] + self.alpha[j] - self.C)
                        H = min(self.C, self.alpha[i] + self.alpha[j])
                    if L == H:
                        continue

                    eta = 2 * K[i, j] - K[i, i] - K[j, j]
                    if eta >= 0:
                        continue

                    self.alpha[j] -= y[j] * (error_i - error_j) / eta
                    self.alpha[j] = np.clip(self.alpha[j], L, H)

                    if np.abs(self.alpha[j] - alpha_j_old) < 1e-5:
                        continue

                    self.alpha[i] += y[i] * y[j] * (alpha_j_old - self.alpha[j])

                    # Compute bias b
                    b1 = self.b - error_i - y[i] * (self.alpha[i] - alpha_i_old) * K[i, i] - y[j] * (self.alpha[j] - alpha_j_old) * K[i, j]
                    b2 = self.b - error_j - y[i] * (self.alpha[i] - alpha_i_old) * K[i, j] - y[j] * (self.alpha[j] - alpha_j_old) * K[j, j]
                    if 0 < self.alpha[i] < self.C:
                        self.b = b1
                    elif 0 < self.alpha[j] < self.C:
                        self.b = b2
                    else:
                        self.b = (b1 + b2) / 2

    def decision_function(self, X):
        # Handle both single and multiple sample cases
        if X.ndim == 1:
            return np.sum([self.alpha[i] * y_train[i] * self.kernel(X_train[i], X, self.degree) for i in range(len(X_train))]) - self.b
        else:
            return np.array([np.sum([self.alpha[i] * y_train[i] * self.kernel(X_train[i], x, self.degree) for i in range(len(X_train))]) - self.b for x in X])

    def predict(self, X):
        return np.sign(self.decision_function(X))

# 使用简化的 SVM 模型
svm = SimplifiedSVM(C=1.0, max_iter=100, kernel=polynomial_kernel, degree=3)
svm.fit(X_train, y_train)

# 进行预测
predictions = svm.predict(X_test)

# 计算准确率
accuracy = np.mean(predictions == y_test)
accuracy



0.5666666666666667